# preprocessing.parsing.ms_office.markitdown.win

> Windows-specific image post-processing around the shared MarkItDown Office workflow.

In [ ]:
# |default_exp preprocessing.parsing.ms_office.markitdown.win

In [ ]:
# | hide
from nbdev.showdoc import *

## Shared API

The platform-neutral conversion and extraction workflow lives in `preprocessing.parsing.ms_office.markitdown.utils`.

In [ ]:
# | export
import os
import re
import asyncio
import shutil
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory

from tqdm.auto import tqdm

from ribosome.preprocessing.parsing.ms_office.markitdown import utils as _utils

ENV_FILE = _utils.ENV_FILE
DEFAULT_MAX_CONCURRENCY = _utils.DEFAULT_MAX_CONCURRENCY
HTML_DATA_IMAGE_RE = _utils.HTML_DATA_IMAGE_RE
IMAGE_EXTENSION_BY_MIME = _utils.IMAGE_EXTENSION_BY_MIME
MARKDOWN_DATA_IMAGE_RE = _utils.MARKDOWN_DATA_IMAGE_RE
OFFICE_EXTENSIONS = _utils.OFFICE_EXTENSIONS
PROJ_ROOT = _utils.PROJ_ROOT
convert_office_to_md = _utils.convert_office_to_md
extract_base64_from_md = _utils.extract_base64_from_md
extract_base64_images = _utils.extract_base64_images
extract_md_base64_images_win = _utils.extract_base64_images
get_office_files_root = _utils.get_office_files_root
process_office_files = _utils.process_office_files


## Windows ImageMagick helpers

In [ ]:
# | export
_GIF_IMAGE_RE = re.compile(
    r"(?P<prefix>!\[[^\]]*\]\()(?P<path>[^)\n]+?\.gif)(?P<suffix>\))",
    flags=re.IGNORECASE,
)


def _visible_markdown_files(root: Path) -> list[Path]:
    """Find Markdown files without descending results from dot-prefixed folders."""
    return sorted(
        path
        for path in root.rglob("*.md")
        if path.is_file()
        and not any(
            part.startswith(".") for part in path.relative_to(root).parts[:-1]
        )
    )


def _magick_command(
    source: Path,
    target: Path,
    *,
    force_opaque: bool = False,
) -> list[str]:
    """Build an ImageMagick command with vector density before its input."""
    command = [
        "magick",
        "-units",
        "PixelsPerInch",
        "-density",
        "300",
        str(source),
    ]
    if force_opaque:
        command.extend(("-background", "white", "-alpha", "off"))
    command.extend([
        "-trim",
        "-border",
        "5",
        str(target),
    ])
    return command


def _opaque_png_command(source: Path, target: Path) -> list[str]:
    """Discard faulty metafile alpha while retaining its rendered RGB strokes."""
    return [
        "magick",
        str(source),
        "-background",
        "white",
        "-alpha",
        "off",
        str(target),
    ]


def _command_error_text(error: Exception) -> str:
    stderr = getattr(error, "stderr", None)
    if stderr:
        lines = [line.strip() for line in str(stderr).splitlines() if line.strip()]
        if lines:
            return lines[-1]
    return str(error)


async def _magick_convert(source: Path, target: Path) -> None:
    """Convert one image with the Windows ImageMagick executable."""
    try:
        await _utils._run_subprocess(_magick_command(source, target))
    except (OSError, subprocess.CalledProcessError) as error:
        tqdm.write(
            f"FAILED: {source} -> {target}: {_command_error_text(error)}"
        )
        raise
    tqdm.write(f"SUCCESS: {source} -> {target}")


def _metafile_suffix_from_header(source: Path) -> str:
    """Detect EMF/WMF data even when an extracted extension is incorrect."""
    with source.open("rb") as input_file:
        header = input_file.read(44)
    if len(header) >= 44 and header[40:44] == b" EMF":
        return ".emf"
    if header.startswith(b"\xd7\xcd\xc6\x9a") or (
        len(header) >= 4
        and header[:2] in {b"\x01\x00", b"\x02\x00"}
        and header[2:4] == b"\x09\x00"
    ):
        return ".wmf"
    return source.suffix.lower()


async def _inkscape_convert_metafile(
    inkscape: str,
    source: Path,
    target: Path,
) -> None:
    """Convert a metafile with Inkscape, normalizing mislabeled inputs."""
    with TemporaryDirectory(prefix="ribosome-metafile-") as temp_dir:
        temporary_root = Path(temp_dir)
        detected_suffix = await asyncio.to_thread(
            _metafile_suffix_from_header,
            source,
        )
        conversion_source = source
        if detected_suffix != source.suffix.lower():
            conversion_source = temporary_root / f"source{detected_suffix}"
            await asyncio.to_thread(shutil.copyfile, source, conversion_source)

        temporary_png = temporary_root / "converted.png"
        await _utils._run_subprocess(
            [
                inkscape,
                str(conversion_source),
                "--export-type=png",
                f"--export-filename={temporary_png}",
                "--export-dpi=300",
                "--export-area-drawing",
            ]
        )
        if not temporary_png.is_file() or temporary_png.stat().st_size == 0:
            raise OSError(
                f"Inkscape did not create a PNG for {source}"
            )
        await asyncio.to_thread(target.parent.mkdir, parents=True, exist_ok=True)
        await _utils._run_subprocess(_opaque_png_command(temporary_png, target))
        if not target.is_file() or target.stat().st_size == 0:
            raise OSError(f"ImageMagick did not normalize {source}")
    tqdm.write(f"SUCCESS (Inkscape): {source} -> {target}")


def _find_inkscape() -> str | None:
    """Find Inkscape on PATH or in its standard Windows install folders."""
    executable = shutil.which("inkscape") or shutil.which("inkscape.com")
    if executable:
        return executable

    for environment_variable in ("ProgramFiles", "ProgramFiles(x86)"):
        program_files = os.environ.get(environment_variable)
        if not program_files:
            continue
        for executable_name in ("inkscape.exe", "inkscape.com"):
            candidate = Path(program_files) / "Inkscape" / "bin" / executable_name
            if candidate.is_file():
                return str(candidate)
    return None


async def _convert_metafile_to_png(source: Path, target: Path) -> None:
    """Convert WMF/EMF to PNG with Inkscape, then ImageMagick fallback."""
    errors = []
    inkscape = _find_inkscape()
    if inkscape:
        try:
            await _inkscape_convert_metafile(inkscape, source, target)
            return
        except (OSError, subprocess.CalledProcessError) as error:
            errors.append(f"Inkscape: {_command_error_text(error)}")

    try:
        await _utils._run_subprocess(
            _magick_command(source, target, force_opaque=True)
        )
    except (OSError, subprocess.CalledProcessError) as error:
        errors.append(f"ImageMagick: {_command_error_text(error)}")
        raise OSError(
            f"Could not convert metafile {source}: {'; '.join(errors)}"
        ) from error
    tqdm.write(f"SUCCESS (ImageMagick): {source} -> {target}")


async def convert_md_gif2png_win(
    markdown_file_path: Path | str,
    image_output_folder: Path | str = ".",
) -> int:
    """Convert linked GIF images to PNG and rewrite their Markdown links."""
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        print(f"Error: Markdown file not found at {markdown_file}")
        return -1

    content = await asyncio.to_thread(markdown_file.read_text, encoding="utf-8")
    replacements = []
    for match in _GIF_IMAGE_RE.finditer(content):
        linked_path = Path(match.group("path"))
        gif_file = (
            linked_path
            if linked_path.is_absolute()
            else markdown_file.parent / linked_path
        )
        if not gif_file.is_file():
            fallback = markdown_file.parent / image_output_folder / linked_path.name
            if fallback.is_file():
                gif_file = fallback
        png_file = gif_file.with_suffix(".png")
        await _magick_convert(gif_file, png_file)
        relative_png = Path(os.path.relpath(png_file, markdown_file.parent)).as_posix()
        replacements.append(
            f'{match.group("prefix")}{relative_png}{match.group("suffix")}'
        )

    if not replacements:
        return -1
    replacement_iter = iter(replacements)
    rewritten = _GIF_IMAGE_RE.sub(lambda _match: next(replacement_iter), content)
    await asyncio.to_thread(markdown_file.write_text, rewritten, encoding="utf-8")
    return len(replacements)


async def convert_gif2png_from_md(
    root_folder: Path | str,
    *,
    show_progress: bool = True,
    max_concurrency: int = DEFAULT_MAX_CONCURRENCY,
) -> dict[str, list]:
    """Concurrently convert linked GIF images in Markdown files below root."""
    root = Path(root_folder).expanduser().resolve()
    report = {"converted": [], "skipped": [], "failed": []}
    markdown_files = await asyncio.to_thread(_visible_markdown_files, root)
    async def convert_one(markdown_file: Path) -> tuple[str, object]:
        try:
            count = await convert_md_gif2png_win(markdown_file, "img")
        except (OSError, subprocess.CalledProcessError) as error:
            tqdm.write(f"Failed to convert {markdown_file}: {error}")
            return "failed", (markdown_file, error)
        if count < 0:
            return "skipped", markdown_file
        return "converted", (markdown_file, count)

    results = await _utils._run_concurrently(
        markdown_files,
        convert_one,
        max_concurrency=max_concurrency,
        description="Converting GIF images",
        unit="file",
        show_progress=show_progress,
    )
    for status, payload in results:
        report[status].append(payload)
    return report


In [ ]:
# | export
_MARKDOWN_VECTOR_IMAGE_RE = re.compile(
    r"(?P<prefix>!\[[^\]]*\]\(\s*<?)"
    r"(?P<path>[^)\n<>]+?\.(?P<suffix>wmf|emf|svg))"
    r"(?P<tail>(?:[?#][^)\n<>]*)?>?(?:\s+[\"'][^)\n]*[\"'])?\s*\))",
    flags=re.IGNORECASE,
)
_HTML_VECTOR_IMAGE_RE = re.compile(
    r"(?P<prefix><img\b[^>]*?\bsrc\s*=\s*(?P<quote>[\"']))"
    r"(?P<path>[^\"']+?\.(?P<suffix>wmf|emf|svg))"
    r"(?P<tail>(?:[?#][^\"']*)?(?P=quote)[^>]*>)",
    flags=re.IGNORECASE,
)


async def _restore_original_image(source: Path, original_data: bytes) -> None:
    """Restore a linked source image if an external converter changed it."""
    try:
        current_data = await asyncio.to_thread(source.read_bytes)
    except FileNotFoundError:
        current_data = None
    if current_data != original_data:
        await asyncio.to_thread(source.parent.mkdir, parents=True, exist_ok=True)
        await asyncio.to_thread(source.write_bytes, original_data)
        tqdm.write(f"RESTORED ORIGINAL: {source}")


async def extract_md_html_images_win(markdown_file_path: Path | str) -> int:
    """Convert linked vectors to PNG while preserving every original file."""
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        return -1

    content = await asyncio.to_thread(markdown_file.read_text, encoding="utf-8")
    matches = sorted(
        (
            *list(_MARKDOWN_VECTOR_IMAGE_RE.finditer(content)),
            *list(_HTML_VECTOR_IMAGE_RE.finditer(content)),
        ),
        key=lambda match: match.start(),
    )
    if not matches:
        return -1

    rewritten_parts = []
    previous_end = 0
    for match in matches:
        linked_path = Path(match.group("path"))
        source = (
            linked_path
            if linked_path.is_absolute()
            else markdown_file.parent / linked_path
        )
        if not source.is_file():
            raise FileNotFoundError(f"Linked vector image not found: {source}")

        original_data = await asyncio.to_thread(source.read_bytes)
        png_file = source.with_suffix(".png")
        try:
            if source.suffix.lower() in {".wmf", ".emf"}:
                await _convert_metafile_to_png(source, png_file)
            else:
                await _magick_convert(source, png_file)
        finally:
            await _restore_original_image(source, original_data)
        relative_png = Path(os.path.relpath(png_file, markdown_file.parent)).as_posix()
        rewritten_parts.extend((
            content[previous_end:match.start()],
            f'{match.group("prefix")}{relative_png}{match.group("tail")}'
        ))
        previous_end = match.end()

    rewritten_parts.append(content[previous_end:])
    rewritten = "".join(rewritten_parts)
    await asyncio.to_thread(markdown_file.write_text, rewritten, encoding="utf-8")
    return len(matches)


async def convert_html_wmf_emf_image_from_md(
    root_folder: Path | str,
    *,
    show_progress: bool = True,
    max_concurrency: int = DEFAULT_MAX_CONCURRENCY,
) -> dict[str, list]:
    """Convert linked vectors below root without removing original files."""
    root = Path(root_folder).expanduser().resolve()
    report = {"converted": [], "skipped": [], "failed": []}
    markdown_files = await asyncio.to_thread(_visible_markdown_files, root)
    async def convert_one(markdown_file: Path) -> tuple[str, object]:
        try:
            count = await extract_md_html_images_win(markdown_file)
        except (OSError, subprocess.CalledProcessError) as error:
            tqdm.write(f"Failed to convert {markdown_file}: {error}")
            return "failed", (markdown_file, error)
        if count < 0:
            return "skipped", markdown_file
        return "converted", (markdown_file, count)

    results = await _utils._run_concurrently(
        markdown_files,
        convert_one,
        max_concurrency=max_concurrency,
        description="Converting WMF/EMF/SVG images",
        unit="file",
        show_progress=show_progress,
    )
    for status, payload in results:
        report[status].append(payload)
    return report


## Windows output-tree helper

In [ ]:
# | export
def copy_md_files(
    src_md_root: Path,
    dst_md_root: Path,
    bOverwrite: bool = True,
) -> dict[str, list]:
    """Copy per-document Markdown folders into another mirrored tree."""
    src_root = src_md_root.expanduser().resolve()
    dst_root = dst_md_root.expanduser().resolve()
    dst_root.mkdir(parents=True, exist_ok=True)
    report = {"copied": [], "skipped": [], "failed": []}

    for markdown_file in sorted(
        path for path in src_root.rglob("*.md") if path.is_file()
    ):
        src_folder = markdown_file.parent
        dst_folder = dst_root / src_folder.relative_to(src_root)
        try:
            if dst_folder.exists():
                if not bOverwrite:
                    report["skipped"].append(dst_folder)
                    continue
                shutil.rmtree(dst_folder)
            shutil.copytree(src_folder, dst_folder)
        except OSError as error:
            report["failed"].append((src_folder, error))
            continue
        report["copied"].append(dst_folder)
    return report


## Run on Windows

The shared workflow and Windows image conversions use bounded asyncio concurrency. Set `MARKITDOWN_MAX_CONCURRENCY` before importing the module (default: `4`), or pass `max_concurrency` explicitly.

In [ ]:
# Run the complete workflow using OFFICE_FILES_ROOT from PROJ_ROOT/.env.
PROCESS_REPORT = await process_office_files(
    overwrite=False,
    image_output_folder="img",
    max_concurrency=DEFAULT_MAX_CONCURRENCY,
)
{
    "office_files_found": len(PROCESS_REPORT["conversion"]["discovered"]),
    "markdown_converted": len(PROCESS_REPORT["conversion"]["converted"]),
    "markdown_skipped": len(PROCESS_REPORT["conversion"]["skipped"]),
    "conversion_failures": len(PROCESS_REPORT["conversion"]["failed"]),
    "images_extracted": PROCESS_REPORT["extraction"]["images_extracted"],
    "extraction_failures": len(PROCESS_REPORT["extraction"]["failed"]),
    "output_root": str(PROCESS_REPORT["conversion"]["output_root"]),
}

In [ ]:
await convert_gif2png_from_md(
    await get_office_files_root(),
    max_concurrency=DEFAULT_MAX_CONCURRENCY,
)

In [ ]:

await convert_html_wmf_emf_image_from_md(
    PROCESS_REPORT["conversion"]["output_root"],
    max_concurrency=DEFAULT_MAX_CONCURRENCY,
)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()